# Superstore Data Cleaning and Preprocessing 

This notebook performs data extraction, cleaning, standardization, and optimization on the Superstore 2019. The goal is to prepare the raw Excel data for analysis by handling missing values, standardizing text formatting, detecting outliers, and optimizing memory usage before saving the final dataset as a Parquet .

In [11]:
# Import the pandas data manipulation
import pandas as pd

## 1. Data Conversion

To improve read performance and compatibility, we first read the original Excel format and convert it into a flat CSV file.

In [12]:
# Read the raw Excel file into a pandas
df = pd.read_excel("Sample - Superstore 2019.xls")

def export_to_CSV(df,path):
    # Export the DataFrame to a CSV format without keeping the index column
    df.to_csv(f"{path.strip()}.csv", index=False)
    
export_to_CSV(df,"Sample - Superstore 2019")

## 2. Load CSV and Parse Dates
Now we load the newly created CSV file. We explicitly parse the date columns to ensure they are treated as datetime objects rather than strings, which is crucial for time-series analysis.

In [13]:
# Read the CSV file, parsing the 'Order Date' and 'Ship Date' columns as datetime64
df = pd.read_csv("Sample - Superstore 2019.csv",parse_dates=["Order Date", "Ship Date"])

# Display the first 5 rows of the dataset to verify it loaded correctly
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2018-138688,2018-06-12,2018-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2017-108966,2017-10-11,2017-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2017-108966,2017-10-11,2017-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 3. Drop Unnecessary Columns
We remove columns that do not provide analytical value, such as arbitrary index identifiers[cite: 2].

In [14]:

def Delete_Columns(df, columns):
    # Creates a copy of the dataframe
    df_clean= df.copy()

    # Drop the listed columns in place
    df_clean.drop(columns = columns, inplace = True)

    # Show the dimensions of the dataframe after dropping columns
    display(df_clean.shape)

    return df_clean

df = Delete_Columns(df, ["Row ID"])

(9994, 20)

## 4. Initial Memory Profiling
Before applying memory optimization techniques, we check the baseline memory usage of our DataFrame

In [15]:
# Display dataframe information including a deep scan of memory usage by object types
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order ID        9994 non-null   object        
 1   Order Date      9994 non-null   datetime64[ns]
 2   Ship Date       9994 non-null   datetime64[ns]
 3   Ship Mode       9994 non-null   object        
 4   Customer ID     9994 non-null   object        
 5   Customer Name   9994 non-null   object        
 6   Segment         9994 non-null   object        
 7   Country/Region  9994 non-null   object        
 8   City            9994 non-null   object        
 9   State           9994 non-null   object        
 10  Postal Code     9983 non-null   float64       
 11  Region          9994 non-null   object        
 12  Product ID      9994 non-null   object        
 13  Category        9994 non-null   object        
 14  Sub-Category    9994 non-null   object        
 15  Prod

## 5. Standardize Categorical Text
To avoid grouping errors due to inconsistent casing or leading/trailing spaces, we clean all object (string) columns.

In [16]:
def Standardize_columns(df):
    df_clean= df.copy()

    # Identify all columns with the 'object' data type (typically strings)
    Standardization = df_clean.select_dtypes(include="object").columns.tolist()

    # Iterate through each string column
    for col in Standardization: 
        # Convert to string, remove leading/trailing spaces, and capitalize the first letter of each word
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
        
    return df_clean

df = Standardize_columns(df)

## 6. Null Handling
We inspect the dataset for missing values. In this dataset, the `Postal Code` column contains nulls for Burlington, Vermont. We impute these with the correct postal code (05401)

In [17]:
def Null_Handling(df):

    df_clean = df.copy()

    # Display a count of missing values for each column
    display(df_clean.isnull().sum())

    # Filter and display rows where 'Postal Code' is missing
    missing_postal = df_clean[df_clean["Postal Code"].isnull()]
    display(missing_postal[["Country/Region", "State", "City", "Postal Code"]])

    # Fill missing postal codes with '05401'  and cast the column to string
    df_clean["Postal Code"] = df_clean["Postal Code"].fillna("05401").astype(str)

    # Display the null counts again to confirm the fix
    display(df_clean.isnull().sum())
    
    return df_clean

df = Null_Handling(df)

Order ID           0
Order Date         0
Ship Date          0
Ship Mode          0
Customer ID        0
Customer Name      0
Segment            0
Country/Region     0
City               0
State              0
Postal Code       11
Region             0
Product ID         0
Category           0
Sub-Category       0
Product Name       0
Sales              0
Quantity           0
Discount           0
Profit             0
dtype: int64

,Country/Region,State,City,Postal Code
2234,United States,Vermont,Burlington,NaN
5274,United States,Vermont,Burlington,NaN
8798,United States,Vermont,Burlington,NaN
9146,United States,Vermont,Burlington,NaN
9147,United States,Vermont,Burlington,NaN
9148,United States,Vermont,Burlington,NaN
9386,United States,Vermont,Burlington,NaN
9387,United States,Vermont,Burlington,NaN
9388,United States,Vermont,Burlington,NaN
9389,United States,Vermont,Burlington,NaN


Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State             0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0
dtype: int64

## 7. Outlier Detection (IQR Method)
We analyze numerical columns using the Interquartile Range (IQR) method to identify potential outliers in Sales, Quantity, Discount, and Profit

In [ ]:
def detect_outliers_iqr(df):

    df_clean = df.copy()
    summary = []

    # Select only numeric columns
    numeric_cols = df_clean.select_dtypes(include= "number").columns.tolist()

    for col in numeric_cols:

        # Calculate the 25th (Q1) and 75th (Q3) percentiles
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)

        # Calculate the Interquartile Range
        IQR = Q3 - Q1

        # Define the lower and upper bounds for outlier detection
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Identify rows that fall outside the bounds
        outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]

        # Append the findings to the summary list
        summary.append({
            "Feature": col,
            "Outlier_Count": len(outliers),
            "Outlier_Pct (%)": (len(outliers) / len(df_clean)) * 100,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound,
            "Min_Value" : df_clean[col].min(),
            "Max_Value" : df_clean[col].max()
        })
    
    return pd.DataFrame(summary)
detect_outliers_iqr(df)

,Feature,Outlier_Count,Outlier_Pct (%),Lower_Bound,Upper_Bound,Min_Value,Max_Value
0,Sales,1167,11.677006,-271.710000,498.930000,0.444,22638.480
1,Quantity,170,1.701021,-2.500000,9.500000,1.000,14.000
2,Discount,856,8.565139,-0.300000,0.500000,0.000,0.800
3,Profit,1881,18.821293,-39.724125,70.816875,-6599.978,8399.976


## 8. Data Type Optimization

To drastically reduce memory usage and speed up future processing, we downcast data types. String columns with low cardinality (fewer than 500 unique values) are converted to the `category` data type, and the remaining objects are explicitly cast to `string`.

In [19]:
def df_clean_conversion(df):

    df_clean = df.copy()

    # Print memory usage before conversion
    df_clean.info(memory_usage="deep")

    # Identify columns with fewer than 500 unique values to convert to categorical types
    category_cols = df_clean[[col for col in df_clean.columns if df_clean[col].nunique() < 500]].columns.tolist()

    for col in category_cols:
            df_clean[col] = df_clean[col].astype("category")

    # Explicitly convert remaining object types to Pandas string type
    str_cols = df_clean.select_dtypes(include="object").columns.tolist()

    for col in str_cols:
            df_clean[col] = df_clean[col].astype("string")

    # Ensure Discount is a 64-bit float and Quantity is a 64-bit integer
    df_clean["Discount"] = df_clean["Discount"].astype("float64")
    df_clean["Quantity"] = df_clean["Quantity"].astype("int64")

    # Print memory usage after conversion to demonstrate the optimization
    df_clean.info(memory_usage="deep")

    return df_clean

df = df_clean_conversion(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order ID        9994 non-null   object        
 1   Order Date      9994 non-null   datetime64[ns]
 2   Ship Date       9994 non-null   datetime64[ns]
 3   Ship Mode       9994 non-null   object        
 4   Customer ID     9994 non-null   object        
 5   Customer Name   9994 non-null   object        
 6   Segment         9994 non-null   object        
 7   Country/Region  9994 non-null   object        
 8   City            9994 non-null   object        
 9   State           9994 non-null   object        
 10  Postal Code     9994 non-null   object        
 11  Region          9994 non-null   object        
 12  Product ID      9994 non-null   object        
 13  Category        9994 non-null   object        
 14  Sub-Category    9994 non-null   object        
 15  Prod

## 9. Save as Parquet
Finally, we export the fully cleaned and optimized DataFrame to the Parquet format. Parquet is highly efficient for analytical queries and preserves our categorical and datetime data types perfectly.

In [20]:
def Save_as_parquet(df,Name):    
    
    #Saves the DataFrame to a Parquet file using the pyarrow engine.
    df.to_parquet(f"{Name.strip()}.parquet", engine="pyarrow")

Save_as_parquet(df, "df_cleaned")